# Logistic regression for loan default

Compare feature sets, regularization, and classification thresholds using the shared partitions from [03_feature_engineering.ipynb](03_feature_engineering.ipynb). Depth-four decision trees serve as benchmarks for the ablation study.

The controlled comparison keeps the same tabular predictors and compares tabular-only logistic regression with Qwen3.5 sentiment and strain scores in both thinking modes.

## 1. Load the shared training and test data

In [1]:
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import norm
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, auc, brier_score_loss, confusion_matrix, f1_score,
    log_loss, precision_recall_curve, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / 'data' / 'financial_loan_dataset_v2.csv').is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError('Open the notebook from the project folder or its notebooks directory.')
DATA_DIR = ROOT / 'data'
TARGET = 'loan_default'
SEED = 42

split_paths = [DATA_DIR / filename for filename in ('train.csv', 'test.csv')]
if not all(path.is_file() for path in split_paths):
    raise FileNotFoundError('Run 03_feature_engineering.ipynb to generate train.csv and test.csv.')

train_df, test_df = (pd.read_csv(path) for path in split_paths)
y_train, y_test = train_df[TARGET], test_df[TARGET]
print(f'Train: {len(y_train):,} rows | Test: {len(y_test):,} rows')

Train: 960 rows | Test: 240 rows


## 2. Define the feature sets and check multicollinearity

Compare raw tabular predictors, engineered financial predictors, and model-generated sentiment and strain scores. VIF measures how well the other predictors explain each column. For the curated set, fit those diagnostic regressions on the training partition.

The all-feature comparison includes a linear dependency introduced by the 36-month burden proxy. The curated predictor set excludes total_burden_ratio and is used for coefficient interpretation.

In [3]:
raw_tabular_cols = [
    'age', 'annual_income', 'credit_score', 'loan_amount',
    'debt_to_income_ratio', 'late_payments_12m', 'account_tenure_months',
    'emp_Full-Time', 'emp_Part-Time', 'emp_Self-Employed'
]
domain_engineered_cols = [
    'age', 'log_annual_income', 'credit_score', 'log_loan_amount',
    'debt_to_income_ratio', 'late_payments_12m', 'log_tenure_months',
    'loan_to_income_ratio', 'total_burden_ratio', 'residual_monthly_income',
    'late_payment_intensity', 'has_late_payments', 'chronic_delinquency_flag',
    'credit_utilization_stress', 'risk_adjusted_leverage',
    'emp_Full-Time', 'emp_Part-Time', 'emp_Self-Employed'
]
curated_tabular_cols = [
    'age', 'log_annual_income', 'credit_score', 'loan_to_income_ratio',
    'debt_to_income_ratio', 'late_payments_12m', 'log_tenure_months',
    'emp_Full-Time', 'emp_Part-Time', 'emp_Self-Employed'
]
nonthinking_text_cols = ['slm_sentiment_score', 'slm_strain_score']
thinking_text_cols = ['slm_thinking_sentiment_score', 'slm_thinking_strain_score']
lr_vif_pruned_cols = curated_tabular_cols + nonthinking_text_cols
unpruned_all_cols = domain_engineered_cols + nonthinking_text_cols


In [4]:
def calculate_vif(features):
    values = {}
    for column in features:
        remaining = features.drop(columns=column)
        model = LinearRegression().fit(remaining, features[column])
        r_squared = model.score(remaining, features[column])
        values[column] = np.inf if r_squared >= 1 else 1 / (1 - r_squared)
    return pd.Series(values, name='VIF').rename_axis('Feature')

vif = calculate_vif(train_df[lr_vif_pruned_cols])
display(vif.sort_values(ascending=False).round(2).to_frame())

,VIF
Feature,
emp_Full-Time,2.69
slm_sentiment_score,2.31
slm_strain_score,2.25
debt_to_income_ratio,2.18
loan_to_income_ratio,2.18
emp_Self-Employed,2.12
emp_Part-Time,1.91
credit_score,1.59
late_payments_12m,1.47


## 3. Compare model configurations

Logistic regression models the default probability as $p=1/(1+\exp(-(\beta_0+\mathbf{x}^{\top}\boldsymbol{\beta})))$. Inverse regularization strength $C$ controls the L1 or L2 penalty. Balanced class weights are included as one experiment. Scaling is part of the pipeline, so each cross-validation fold learns its own scaling parameters.

The paired configurations use the same curated tabular predictors, L2 regularization with C = 0.5, and threshold 0.50. The tabular-only baseline isolates the effect of adding the two Qwen3.5 scores. The two Qwen rows differ by the thinking mode used to extract those scores.

The final configuration uses the same fitted non-thinking model as the 0.50 row. Its threshold maximizes macro F1 on five-fold out-of-fold training predictions and is then applied to the held-out test set.

In [5]:
def logistic_pipeline(c_value, penalty, class_weight=None):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=c_value,
            penalty=penalty,
            class_weight=class_weight,
            solver='liblinear' if penalty == 'l1' else 'lbfgs',
            max_iter=2000,
            random_state=SEED,
        ),
    )

comparison_c = 0.5
comparison_model = logistic_pipeline(comparison_c, 'l2')
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
training_probabilities = cross_val_predict(
    comparison_model,
    train_df[lr_vif_pruned_cols],
    y_train,
    cv=folds,
    method='predict_proba',
)[:, 1]
thresholds = np.arange(5, 96) / 100
oof_scores = np.array([
    f1_score(y_train, training_probabilities >= threshold, average='macro')
    for threshold in thresholds
])
best_index = oof_scores.argmax()
tuned_threshold = thresholds[best_index]
print(f'Training-selected threshold: {tuned_threshold:.2f}')
print(f'Out-of-fold macro F1: {oof_scores[best_index]:.4f}')

Training-selected threshold: 0.27
Out-of-fold macro F1: 0.7243


## 4. Evaluate on the held-out test set

Accuracy, F1, and confusion-matrix counts use each row's threshold. ROC AUC, trapezoidal precision–recall AUC, Brier score, and log loss use the predicted probabilities.

In [6]:
def evaluate_predictions(name, y_true, probabilities, threshold=0.50):
    predicted = probabilities >= threshold
    tn, fp, fn, tp = confusion_matrix(y_true, predicted, labels=[0, 1]).ravel()
    precision, recall, _ = precision_recall_curve(y_true, probabilities)
    return {
        'Model': name,
        'Threshold': threshold,
        'Accuracy': accuracy_score(y_true, predicted),
        'Precision': precision_score(y_true, predicted, zero_division=0),
        'Recall': recall_score(y_true, predicted),
        'Specificity': tn / (tn + fp),
        'F1_Score': f1_score(y_true, predicted),
        'Macro_F1': f1_score(y_true, predicted, average='macro'),
        'ROC_AUC': roc_auc_score(y_true, probabilities),
        'PR_AUC': auc(recall, precision),
        'Brier_Score': brier_score_loss(y_true, probabilities),
        'Log_Loss': log_loss(y_true, probabilities),
        'TP': tp, 'TN': tn, 'FP': fp, 'FN': fn,
    }

In [7]:
configs = [
    ('1. Logistic regression, raw tabular', raw_tabular_cols, 1.0, 'l2', None),
    ('2. Logistic regression, financial ratios and logs', domain_engineered_cols, 1.0, 'l2', None),
    ('3. All engineered predictors, including redundant ratios', unpruned_all_cols, 1.0, 'l2', None),
    ('4. Curated Qwen non-thinking, balanced weights', lr_vif_pruned_cols, 1.0, 'l2', 'balanced'),
    ('5. Curated Qwen non-thinking, L1', lr_vif_pruned_cols, comparison_c, 'l1', None),
    ('6a. Curated tabular only', curated_tabular_cols, comparison_c, 'l2', None),
    ('6b. Curated + Qwen thinking (32 tokens)', curated_tabular_cols + thinking_text_cols, comparison_c, 'l2', None),
]

results = []
for name, features in [
    ('0a. Decision tree, raw tabular', raw_tabular_cols),
    ('0b. Decision tree, engineered + Qwen non-thinking', unpruned_all_cols),
]:
    tree = DecisionTreeClassifier(max_depth=4, random_state=SEED)
    tree.fit(train_df[features], y_train)
    probabilities = tree.predict_proba(test_df[features])[:, 1]
    results.append(evaluate_predictions(name, y_test, probabilities))

for name, features, c_value, penalty, class_weight in configs:
    model = logistic_pipeline(c_value, penalty, class_weight)
    model.fit(train_df[features], y_train)
    probabilities = model.predict_proba(test_df[features])[:, 1]
    results.append(evaluate_predictions(name, y_test, probabilities))

comparison_model.fit(train_df[lr_vif_pruned_cols], y_train)
test_probabilities = comparison_model.predict_proba(test_df[lr_vif_pruned_cols])[:, 1]
for name, threshold in [
    ('6c. Curated + Qwen non-thinking', 0.50),
    ('6d. Curated + Qwen non-thinking, tuned threshold', tuned_threshold),
]:
    results.append(evaluate_predictions(name, y_test, test_probabilities, threshold))

benchmark = pd.DataFrame(results)
display(benchmark.round(4))

results_dir = ROOT / 'results'
results_dir.mkdir(exist_ok=True)
benchmark.to_csv(results_dir / 'logistic_regression_metrics.csv', index=False)


,Model,Threshold,Accuracy,Precision,Recall,Specificity,F1_Score,Macro_F1,ROC_AUC,PR_AUC,Brier_Score,Log_Loss,TP,TN,FP,FN
0,"0a. Decision tree, raw tabular",0.50,0.8083,0.7000,0.14,0.9842,0.2333,0.5619,0.7265,0.4070,0.1481,0.7468,7,187,3,43
1,"0b. Decision tree, engineered + Qwen non-thinking",0.50,0.8208,0.7333,0.22,0.9789,0.3385,0.6174,0.7115,0.5219,0.1380,0.4506,11,186,4,39
2,"1. Logistic regression, raw tabular",0.50,0.8208,0.6400,0.32,0.9526,0.4267,0.6602,0.8144,0.5216,0.1312,0.4055,16,181,9,34
3,"2. Logistic regression, financial ratios and logs",0.50,0.8083,0.5909,0.26,0.9526,0.3611,0.6242,0.7959,0.4979,0.1382,0.4268,13,181,9,37
4,"3. All engineered predictors, including redund...",0.50,0.8250,0.6538,0.34,0.9526,0.4474,0.6717,0.7977,0.5224,0.1334,0.4166,17,181,9,33
5,"4. Curated Qwen non-thinking, balanced weights",0.50,0.7542,0.4400,0.66,0.7789,0.5280,0.6809,0.8133,0.5636,0.1625,0.4884,33,148,42,17
6,"5. Curated Qwen non-thinking, L1",0.50,0.8208,0.6522,0.30,0.9579,0.4110,0.6527,0.8127,0.5590,0.1282,0.4009,15,182,8,35
7,6a. Curated tabular only,0.50,0.8167,0.6364,0.28,0.9579,0.3889,0.6405,0.8063,0.5193,0.1327,0.4110,14,182,8,36
8,6b. Curated + Qwen thinking (32 tokens),0.50,0.8208,0.6522,0.30,0.9579,0.4110,0.6527,0.8128,0.5618,0.1282,0.4015,15,182,8,35
9,6c. Curated + Qwen non-thinking,0.50,0.8208,0.6522,0.30,0.9579,0.4110,0.6527,0.8115,0.5554,0.1285,0.4020,15,182,8,35


## 5. Interpret the curated non-thinking L2 model

A coefficient $\beta$ is the change in fitted log-odds for a one-standard-deviation increase in a predictor, holding the other inputs fixed. Its odds ratio is $\exp(\beta)$. For a binary indicator, Odds_Ratio_Per_Unit gives the fitted odds ratio for a change from 0 to 1.

The inverse penalized Hessian gives a local curvature approximation around the fitted coefficients. The resulting Wald quantities are descriptive for this penalized fit and should be interpreted cautiously on this synthetic dataset.

In [8]:
scaler = comparison_model.named_steps['standardscaler']
estimator = comparison_model.named_steps['logisticregression']
scaled_train = scaler.transform(train_df[lr_vif_pruned_cols])
fitted_probabilities = comparison_model.predict_proba(train_df[lr_vif_pruned_cols])[:, 1]
weights = fitted_probabilities * (1 - fitted_probabilities)
design = np.column_stack([scaled_train, np.ones(len(scaled_train))])
penalty_matrix = np.eye(design.shape[1]) / comparison_c
penalty_matrix[-1, -1] = 0
information = (design.T * weights) @ design + penalty_matrix
se = np.sqrt(np.diag(np.linalg.pinv(information)))[:-1]

beta = estimator.coef_[0]
z_score = beta / se
coefficient_table = pd.DataFrame(
    {
        'Beta_Per_SD': beta,
        'Odds_Ratio_Per_SD': np.exp(beta),
        'Odds_Ratio_Per_Unit': np.exp(beta / scaler.scale_),
        'Approx_Std_Error': se,
        'Approx_OR_CI_Lower': np.exp(beta - norm.ppf(0.975) * se),
        'Approx_OR_CI_Upper': np.exp(beta + norm.ppf(0.975) * se),
        'Approx_Wald_Z': z_score,
        'Approx_Wald_P': 2 * norm.sf(np.abs(z_score)),
        'VIF': vif.to_numpy(),
    },
    index=vif.index,
).sort_values('Beta_Per_SD', key=abs, ascending=False)
display(coefficient_table.round(4))

,Beta_Per_SD,Odds_Ratio_Per_SD,Odds_Ratio_Per_Unit,Approx_Std_Error,Approx_OR_CI_Lower,Approx_OR_CI_Upper,Approx_Wald_Z,Approx_Wald_P,VIF
Feature,,,,,,,,,
credit_score,-0.9881,0.3723,0.9871,0.1306,0.2882,0.4809,-7.5660,0.0000,1.5878
late_payments_12m,0.5850,1.7951,1.5518,0.1061,1.4580,2.2100,5.5142,0.0000,1.4683
debt_to_income_ratio,0.4183,1.5194,83.2077,0.1352,1.1657,1.9804,3.0943,0.0020,2.1792
log_tenure_months,-0.2098,0.8108,0.7648,0.0943,0.6740,0.9753,-2.2248,0.0261,1.0210
slm_sentiment_score,-0.2045,0.8151,0.7154,0.1254,0.6374,1.0423,-1.6300,0.1031,2.3107
emp_Part-Time,-0.1681,0.8453,0.5936,0.1269,0.6592,1.0840,-1.3242,0.1854,1.9076
age,0.0812,1.0846,1.0088,0.0994,0.8926,1.3178,0.8168,0.4140,1.0337
slm_strain_score,0.0740,1.0768,1.6304,0.1297,0.8351,1.3886,0.5707,0.5682,2.2486
emp_Full-Time,-0.0668,0.9354,0.8715,0.1481,0.6998,1.2503,-0.4513,0.6518,2.6873
